In [ ]:
import numpy as np

path = "2025_combined_mlar_header."

data = np.genfromtxt(path,
                    delimiter = "|",
                    names = True,
                    # dtype = None,
                    missing_values = ("NA", "")
                    ,max_rows = 9998
                    )

In [ ]:
import os
import zipfile

print([f for f in os.listdir() if f.endswith(".zip")])

In [ ]:
import pandas as pd
import zipfile

In [ ]:
zip_path = "2025_combined_mlar_header.zip"
selected_columns = [
    "state_code",
    "loan_type",
    "action_taken",
    "debt_to_income_ratio"
]
with zipfile.ZipFile(zip_path, "r") as z:
    txt_name = next(
        name for name in z.namelist()
        if name.lower().endswith(".txt")
    )

    with z.open(txt_name) as f:
        extracted_data = pd.read_csv(
            f,
            sep="|",
            usecols=selected_columns,
            nrows=9998,
            dtype=str
        )

extracted_data = extracted_data.rename(columns={
    "state_code": "state",
    "debt_to_income_ratio": "debt_to_income"
})

print(extracted_data.head(10))
print(extracted_data.shape)

In [ ]:
print(extracted_data.isnull().sum())

In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
df_clean = extracted_data.copy()
def convert_dti(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    try:
        return float(value)
    except ValueError:
        pass
    match = re.fullmatch(
        r"(\d+(?:\.\d+)?)%-<(\d+(?:\.\d+)?)%",
        value
    )

    if match:
        lower = float(match.group(1))
        upper = float(match.group(2))
        return (lower + upper) / 2
    if value == ">60%":
        return 65.0
    return np.nan

In [ ]:
df_clean["dti_numeric"] = (
    df_clean["debt_to_income"].apply(convert_dti)
)
print("Missing before:",
      df_clean["dti_numeric"].isna().sum())
df_clean["dti_interpolated"] = (
    df_clean["dti_numeric"].interpolate(
        method="linear",
        limit_area="inside"
    )
)
print("Missing after:",
      df_clean["dti_interpolated"].isna().sum())
print(df_clean[
    ["debt_to_income",
     "dti_numeric",
     "dti_interpolated"]
].head(10))

In [ ]:
data_array = extracted_data.to_numpy()

print(data_array.shape)
print(type(data_array))

In [ ]:
print(data.shape)
print(data.dtype.names)

In [ ]:
selected_data = data[
    ["state_code",
     "loan_type",
     "action_taken",
     "debt_to_income_ratio"]
    ]